# Customers

In [0]:
customers = spark.read.table("workspace.ecompulse_bronze.customers")

In [0]:
customers.display()

In [0]:
customers = customers.drop("customer_unique_id")

In [0]:
customers.describe().show()

In [0]:
customers.count()

In [0]:
customers.printSchema()

In [0]:
customers.write.mode("overwrite").format("delta").saveAsTable("workspace.ecompulse_silver.customers")

- No null values are there.
- `customer_unique_id` has been dropped since it has no real use or connection with any other table.
- records with duplicate `customer_unique_id` exist, therefore decided to drop the column.

# Orders

In [0]:
orders = spark.read.table("workspace.ecompulse_bronze.orders")

In [0]:
orders.display()

In [0]:
import pyspark.sql.functions as F

In [0]:
null_counts = orders.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c)
    for c in orders.columns
]).show()

In [0]:
drop_columns = ["order_approved_at", "order_delivered_carrier_date"]

orders = orders.drop(*drop_columns)

In [0]:
orders = orders.withColumn(
    "order_delivered_customer_date",
    F.when(
        F.col("order_status") == "delivered", 
        F.coalesce(F.col("order_delivered_customer_date"), F.col("order_estimated_delivery_date"))
    ).otherwise(F.col("order_delivered_customer_date"))
)

In [0]:
orders.show(10)

In [0]:
null_counts = orders.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c)
    for c in orders.columns
]).show()

- these nulls are valid since these are for the orders which have not been delivered

In [0]:
orders = orders.withColumns({
    "order_purchase_date": F.to_date(F.col("order_purchase_timestamp")),
    "order_purchase_hour": F.hour(F.col("order_purchase_timestamp")),
    "order_delivered_date": F.to_date(F.col("order_delivered_customer_date")),
    "order_delivered_hour": F.hour(F.col("order_delivered_customer_date"))
})

In [0]:
orders.display()

In [0]:
orders.write.mode("overwrite").saveAsTable("workspace.ecompulse_silver.orders")

# Products

In [0]:
products = spark.read.table("workspace.ecompulse_bronze.products")

In [0]:
products.display()

In [0]:
products.describe().show()

In [0]:
products.count()

In [0]:
null_counts = products.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c)
    for c in products.columns
]).show()

In [0]:
products.filter(
    F.col("product_category_name").isNull() |
    F.col("product_name_lenght").isNull() |
    F.col("product_description_lenght").isNull() |
    F.col("product_photos_qty").isNull()
).select("*")\
.show()

In [0]:
products = products.fillna({
    "product_category_name" : "Unknown",
    "product_name_lenght" : 0,
    "product_description_lenght" : 0,
    "product_photos_qty" : 0
})

In [0]:
products.filter(
    (products["product_weight_g"].isNull()) |
    (products["product_length_cm"].isNull()) |
    (products["product_height_cm"].isNull()) |
    (products["product_width_cm"].isNull())
).show()

In [0]:
products.display()

In [0]:
products.write.mode("overwrite").saveAsTable("workspace.ecompulse_silver.products")